# PRISM VPP Dispatch Demo
#### Real-Time Coordination for Large Distributed Energy Fleets
*GPU-native proprietary optimization engine · NYISO real-data validated · Asymmetry Computing*

Run all cells: **Kernel → Restart Kernel and Run All Cells**


In [ ]:
# Install packages via micropip (runs in-browser via Pyodide)
import sys
_IN_PYODIDE = "pyodide" in sys.modules or hasattr(sys, "_pyodide_core")

if _IN_PYODIDE:
    import micropip
    await micropip.install(["plotly", "pandas", "ipywidgets"])
else:
    # Standard Colab / local Jupyter
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "plotly", "pandas", "ipywidgets"])

import warnings, math, hashlib, datetime as _dt, json as _json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

RNG = np.random.default_rng(42)
print(f"✓ Environment ready  |  Pyodide={_IN_PYODIDE}")


In [ ]:
# ── Demo configuration ──────────────────────────────────────────────────────
DEMO_SPEED = "standard"   # "quick" | "standard" | "full"
DEMO_N     = {"quick": 2_000, "standard": 10_000, "full": 50_000}[DEMO_SPEED]
DEMO_MODE  = "SYNTHETIC"
print(f"Fleet N: {DEMO_N:,}  |  Mode: {DEMO_MODE}")


In [ ]:
_CSS = '<style>:root{--bg:#f6f8fa;--card:#fff;--card2:#f6f8fa;--bdr:#d0d7de;--tx:#1f2328;--tx2:#57606a;--acc:#0969da;--grn:#1a7f37;--amb:#9a6700;--red:#d1242f;}.pcard{background:var(--card);border:1px solid var(--bdr);border-radius:8px;padding:20px 24px;margin:12px 0;font-family:-apple-system,sans-serif;color:var(--tx);box-shadow:0 1px 3px rgba(0,0,0,.06);}.phdr{background:linear-gradient(135deg,#eef4ff,#f0f6ff);border:1px solid #bfdbfe;border-top:3px solid var(--acc);border-radius:8px;padding:28px 36px;margin:16px 0;font-family:-apple-system,sans-serif;}.bdg{display:inline-block;padding:3px 10px;border-radius:20px;font-size:10.5px;font-weight:600;letter-spacing:.04em;font-family:monospace;}.bdg-live{background:#dcfce7;color:#15803d;border:1px solid #86efac;}.bdg-cached{background:#dbeafe;color:#1d4ed8;border:1px solid #93c5fd;}.bdg-synth{background:#ede9fe;color:#7c3aed;border:1px solid #c4b5fd;}.bdg-assum{background:#fef9c3;color:#854d0e;border:1px solid #fde047;}.mgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:10px;margin:12px 0;}.mbox{background:var(--card2);border:1px solid var(--bdr);border-radius:6px;padding:14px;text-align:center;}.mval{font-size:19px;font-weight:700;color:var(--acc);font-family:monospace;line-height:1.2;}.mlbl{font-size:10px;color:var(--tx2);margin-top:4px;text-transform:uppercase;letter-spacing:.05em;}.stitle{font-size:12px;font-weight:700;color:var(--tx2);text-transform:uppercase;letter-spacing:.1em;border-bottom:1px solid var(--bdr);padding-bottom:8px;margin-bottom:14px;}.ipnote{background:#f0f6ff;border:1px solid #bfdbfe;border-left:3px solid var(--acc);border-radius:4px;padding:12px 16px;font-size:12px;color:#57606a;font-family:monospace;margin:12px 0;}.ptbl{width:100%;border-collapse:collapse;font-family:monospace;font-size:12.5px;}.ptbl th{background:var(--card2);border:1px solid var(--bdr);padding:8px 12px;text-align:left;color:var(--tx2);font-size:10.5px;text-transform:uppercase;letter-spacing:.05em;}.ptbl td{border:1px solid #e9ecef;padding:7px 12px;color:var(--tx);}.ptbl tr:hover td{background:var(--card2);}.grn{color:#1a7f37!important;}.amb{color:#9a6700!important;}.red{color:#d1242f!important;}.blu{color:#0969da!important;}</style>'
display(HTML(_CSS))
print("✓ Theme applied")

In [ ]:
# ── Validated cached benchmark data ─────────────────────────────────────────
BENCH = {
    "gpu_scale": dict(
        units=[50_000, 200_000, 500_000],
        prism_s=[1.58, 6.24, 15.79],
        source="PRISM internal benchmark / RTX 4000 Ada / NYISO Jan 2024",
    ),
    "cpu_vs_gurobi": dict(
        units=[500, 2_000, 10_000, 50_000],
        prism_s=[0.32, 1.38, 9.20, 59.53],
        gurobi_s=[0.55, 2.48, 15.72, None],
        source="PRISM internal benchmark / Gurobi Academic / NYISO Jan 2024",
    ),
    "quality":  dict(N=2000, gap_pct=0.004, feas_viol=0.0),
    "realtime": dict(N=10_000, cycles=288, mean_s=0.41, p95_s=0.46, p99_s=0.50),
    "admm":     dict(units=[2_000,10_000], speedup=[11.8,74.5],
                     prism_better_pct=[43.7,84.6]),
    "warmstart":dict(cold_iters=575, cold_s=284.8, warm_iters=338, warm_s=181.7),
    "coordination": dict(
        cap_pct=[100, 60, 45, 30],
        increment_pct=[0.00, 12.75, 19.03, 28.09],
        label=["No constraint","60% cap","45% cap","30% cap (tight)"],
        source="PRISM coordination benchmark / NYISO RT Jan 2024 / N=300 / eta=0.85",
    ),
    "economics": dict(per_mw_yr=61_140, increment_myr=16.0,
                      source="PRISM economic sizing / NYISO RT Jan 2024"),
}
print("✓ Benchmark data loaded:", list(BENCH.keys()))


In [ ]:
# ── Cover ────────────────────────────────────────────────────────────────────
_cover = """
<div class="phdr">
  <div style="color:#57606a;font-size:11px;letter-spacing:.12em;text-transform:uppercase;margin-bottom:10px;">
    Asymmetry Computing &middot; PRISM Engine &middot; VPP / DER Dispatch &middot; Demo v1.0
  </div>
  <h1 style="font-size:26px;font-weight:700;color:#1f2328;margin:0 0 8px;line-height:1.2;">
    Real-Time Coordination for Large<br>Distributed Energy Fleets
  </h1>
  <p style="font-size:14px;color:#57606a;margin:0 0 18px;max-width:680px;line-height:1.7;">
    PRISM is a GPU-native proprietary optimization engine that coordinates 10&#8308;&ndash;10&#8310;
    distributed energy resources inside 5-minute market clearing windows,
    with feasibility certificates on every solve.
  </p>
  <div style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:16px;">
    <span class="bdg bdg-synth">Synthetic demo mode</span>
    <span class="bdg bdg-cached">NYISO real data &middot; Jan 2024</span>
    <span class="bdg" style="background:#dcfce7;color:#15803d;border:1px solid #86efac;">
      Gates G1&ndash;G6 closed &#10003;
    </span>
  </div>
  <div class="ipnote">
    &#9888; IP NOTICE &mdash; This notebook evaluates PRISM through its public input/output
    behaviour only. Internal solver mechanics are intentionally withheld.
    Full methodology available under NDA.
  </div>
</div>
"""
display(HTML(_cover))


In [ ]:
# ── Section 1: Synthetic Fleet Dataset ───────────────────────────────────────
display(HTML(
    '<div class="pcard"><div class="stitle">Section 1 — Demo Fleet Dataset</div>'
    '<span class="bdg bdg-synth">SYNTHETIC</span> '
    'Fleet generated in-browser. Real operator data supplied at pilot via secure API or VPC.'
    '</div>'
))

T = 24
FLEET_MIX = dict(
    BESS    =dict(frac=0.45,pbar=(5,15),  soc=(0.4,0.6),color="#0969da"),
    EV      =dict(frac=0.25,pbar=(3,7),   soc=(0.2,0.8),color="#1a7f37"),
    Solar   =dict(frac=0.15,pbar=(2,8),   soc=(0.0,0.0),color="#9a6700"),
    Wind    =dict(frac=0.05,pbar=(10,30), soc=(0.0,0.0),color="#8250df"),
    FlexLoad=dict(frac=0.07,pbar=(5,50),  soc=(0.0,0.0),color="#0969da"),
    Thermal =dict(frac=0.03,pbar=(50,200),soc=(0.0,0.0),color="#57606a"),
)

records = []
for dtype, cfg in FLEET_MIX.items():
    n = max(1, int(DEMO_N * cfg["frac"]))
    plo, phi = cfg["pbar"]; slo, shi = cfg["soc"]
    pbar = RNG.uniform(plo, phi, n) / 1000
    soc0 = RNG.uniform(slo, shi, n)
    avail = RNG.uniform(0.80, 0.99, n)
    for i in range(n):
        records.append(dict(device_id=f"{dtype[:3]}-{i:06d}",device_type=dtype,
                            pbar_mw=round(float(pbar[i]),4),soc0=round(float(soc0[i]),3),
                            availability=round(float(avail[i]),3),
                            zone=int(RNG.integers(0,15)),feeder=int(RNG.integers(0,50))))

fleet = pd.DataFrame(records)
total_mw = fleet.pbar_mw.sum()
online   = int((fleet.availability >= 0.85).sum())

display(HTML(
    f'<div class="mgrid">'
    f'<div class="mbox"><div class="mval">{len(fleet):,}</div><div class="mlbl">Total devices</div></div>'
    f'<div class="mbox"><div class="mval">{total_mw:.1f}</div><div class="mlbl">Nameplate MW</div></div>'
    f'<div class="mbox"><div class="mval">{online:,}</div><div class="mlbl">Online (avail≥85%)</div></div>'
    f'<div class="mbox"><div class="mval">{len(FLEET_MIX)}</div><div class="mlbl">Resource types</div></div>'
    f'<div class="mbox"><div class="mval">15</div><div class="mlbl">NYISO zones</div></div>'
    f'<div class="mbox"><div class="mval">50</div><div class="mlbl">Feeder segments</div></div>'
    f'</div>'
))

# Fleet composition chart
agg = fleet.groupby("device_type").agg(count=("pbar_mw","count"),mw=("pbar_mw","sum")).reset_index()
col_map = {d:v["color"] for d,v in FLEET_MIX.items()}
fig = make_subplots(rows=1,cols=2,
                    subplot_titles=["Device Count by Type","Nameplate MW by Type"],
                    specs=[[{"type":"pie"},{"type":"bar"}]])
fig.add_trace(go.Pie(labels=agg.device_type,values=agg["count"],
                      marker_colors=[col_map.get(d,"#888") for d in agg.device_type],
                      hole=0.45,textinfo="label+percent",showlegend=False),row=1,col=1)
fig.add_trace(go.Bar(x=agg.device_type,y=agg.mw,
                      marker_color=[col_map.get(d,"#888") for d in agg.device_type],
                      text=agg.mw.round(1),textposition="outside",showlegend=False),row=1,col=2)
fig.update_layout(
    title=dict(text=f"Synthetic Fleet  N={len(fleet):,}  <span style='font-size:11px;color:#8250df'>[SYNTHETIC]</span>",
               font=dict(color="#1f2328",size=13),x=0.01),
    paper_bgcolor="#f6f8fa",plot_bgcolor="#f6f8fa",
    height=300,margin=dict(l=20,r=20,t=55,b=20),
    font=dict(family="monospace",color="#57606a"),
)
fig.update_yaxes(gridcolor="#e9ecef",row=1,col=2)
fig.show()


In [ ]:
# ── Section 2: 5-Minute Market Clearing Window ───────────────────────────────
h = np.arange(T)
base_p = 55+25*np.sin(2*np.pi*(h-7)/24)+40*np.exp(-((h-18)**2)/8)+10*np.exp(-((h-9)**2)/5)
p_obs  = base_p + RNG.normal(0,8,T)
_price = p_obs

fig = go.Figure()
phases = [
    ("Market signal + ingestion", 0,  30, "#4f86f7", 0.75),
    ("PRISM GPU solve",           30, 71, "#1a7f37", 0.85),
    ("Feasibility cert",          71, 75, "#9a6700", 0.9),
    ("Dispatch submission",       75, 90, "#1a7f37", 0.5),
    ("Settlement buffer",         90,300, "#e9ecef", 0.9),
]
for lbl,t0,t1,col,op in phases:
    fig.add_shape(type="rect",x0=t0,x1=t1,y0=0.1,y1=0.9,
                  fillcolor=col,opacity=op,line_width=0)
    if t1-t0>12:
        fig.add_annotation(x=(t0+t1)/2,y=0.5,text=lbl,showarrow=False,
                           font=dict(size=9,color="white",family="monospace"),
                           xanchor="center",yanchor="middle")

fig.add_shape(type="line",x0=300,x1=300,y0=0,y1=1,
              line=dict(color="#d1242f",width=2,dash="dash"))
fig.add_annotation(x=298,y=0.9,text="5-min deadline",showarrow=False,xanchor="right",
                   font=dict(size=9,color="#d1242f",family="monospace"))
fig.add_annotation(x=73,y=0.93,
    text=f"p99 = {BENCH['realtime']['p99_s']}s @ N=10k · {BENCH['realtime']['cycles']} cycles",
    showarrow=False,font=dict(size=9,color="#1a7f37",family="monospace"),xanchor="left")

fig.update_layout(
    title=dict(text="Market Clearing Window  <span style='font-size:11px;color:#1d4ed8'>[CACHED · PRISM real-time cadence benchmark]</span>",
               font=dict(color="#1f2328",size=13),x=0.01),
    xaxis=dict(title="seconds",range=[-5,315],gridcolor="#e9ecef",zeroline=False,color="#57606a"),
    yaxis=dict(visible=False,range=[0,1]),
    paper_bgcolor="#f6f8fa",plot_bgcolor="#f6f8fa",
    height=165,margin=dict(l=20,r=20,t=50,b=40),
    font=dict(family="monospace",color="#57606a"),
)
fig.show()


In [ ]:
# ── Section 3: Coordination Premium — THE KEY RESULT ─────────────────────────
display(HTML(
    '<div class="pcard"><div class="stitle">Section 3 — Coordination Value: The Decisive Test</div>'
    '<span class="bdg bdg-cached">CACHED · PRISM coordination benchmark · NYISO RT Jan 2024</span>'
    '<p style="color:#57606a;font-size:13px;line-height:1.7;margin:10px 0 0;">'
    'Centralized PRISM vs realistic independent dispatch + merit-order curtailment, '
    'swept over feeder export cap tightness. N=300, η=0.85, 1.5 cyc/day.'
    '</p></div>'
))

caps  = BENCH["coordination"]["cap_pct"]
incs  = BENCH["coordination"]["increment_pct"]
lbls  = BENCH["coordination"]["label"]
cols  = ["#d0d7de" if v==0 else ("#6fdd8b" if v<15 else ("#2da44e" if v<22 else "#1a7f37")) for v in incs]

fig = go.Figure(go.Bar(
    x=lbls,y=incs,marker_color=cols,width=0.5,
    text=[f"+{v:.2f}%" for v in incs],
    textposition="outside",
    textfont=dict(size=12,color="#1f2328",family="monospace"),
))
fig.add_annotation(x="No constraint",y=0.8,
    text="0.00% — zero uplift when<br>feeders don't bind (validates test)",
    showarrow=False,font=dict(size=9,color="#57606a",family="monospace"),align="center")
fig.update_layout(
    title=dict(
        text="Coordination Premium: Centralized vs Independent+Curtail  "
             "<span style='font-size:11px;color:#1d4ed8'>[CACHED]</span>",
        font=dict(color="#1f2328",size=13),x=0.01),
    xaxis=dict(gridcolor="#e9ecef",zeroline=False,color="#57606a"),
    yaxis=dict(title="Incremental value vs independent dispatch (%)",
               range=[-2,34],gridcolor="#e9ecef",zeroline=True,
               zerolinecolor="#e9ecef",color="#57606a"),
    paper_bgcolor="#f6f8fa",plot_bgcolor="#f6f8fa",
    height=360,margin=dict(l=50,r=20,t=70,b=20),
    font=dict(family="monospace",color="#57606a"),
    shapes=[dict(type="line",x0=-0.5,x1=3.5,y0=10,y1=10,
                 line=dict(color="#0969da",width=1,dash="dot"))],
    annotations=[dict(x=3.4,y=10.8,text="10% signal threshold",showarrow=False,
                      font=dict(size=9,color="#0969da",family="monospace"),xanchor="right")],
)
fig.show()

display(HTML(
    '<div class="pcard"><div class="mgrid">'
    '<div class="mbox"><div class="mval" style="color:#d0d7de">0.00%</div><div class="mlbl">No constraint</div></div>'
    '<div class="mbox"><div class="mval grn">+12.75%</div><div class="mlbl">60% cap</div></div>'
    '<div class="mbox"><div class="mval grn">+19.03%</div><div class="mlbl">45% cap</div></div>'
    '<div class="mbox"><div class="mval grn">+28.09%</div><div class="mlbl">30% cap (tight)</div></div>'
    '</div>'
    '<p style="color:#57606a;font-size:12px;font-family:monospace;margin-top:10px;">'
    '&#10003; Coordination value is <strong style="color:#1f2328">real and material</strong> when feeders bind.<br>'
    '&#10003; Zero uplift without binding constraint validates test integrity.<br>'
    '&#9888; Converged decomposition reaches the same optimum — PRISM\'s edge is <em>speed</em> (11.8–74.5× faster).'
    '</p></div>'
))


In [ ]:
# ── Section 4: Solver Comparison  [CACHED] ────────────────────────────────────
units_all = [500,2_000,10_000,50_000,200_000,500_000]
prism_gpu = [None,None,None,1.58,6.24,15.79]
prism_cpu = [0.32,1.38,9.20,59.53,None,None]
gurobi_t  = [0.55,2.48,15.72,None,None,None]
admm_t    = [None,10.20,53.68,None,None,None]

def _tr(name,x,y,col,dash="solid"):
    pts=[(xi,yi) for xi,yi in zip(x,y) if yi is not None]
    return go.Scatter(x=[p[0] for p in pts],y=[p[1] for p in pts],
                      mode="lines+markers",name=name,
                      line=dict(color=col,width=2,dash=dash),marker=dict(size=7))

fig = go.Figure([
    _tr("PRISM GPU",       units_all,prism_gpu,"#0969da"),
    _tr("PRISM CPU",       units_all,prism_cpu,"#4f86f7","dot"),
    _tr("Gurobi (OOM≥50k)",units_all,gurobi_t, "#9a6700"),
    _tr("Distributed ADMM",units_all,admm_t,   "#d1242f","dash"),
])
fig.add_hline(y=300,line=dict(color="#1a7f37",width=1.5,dash="dash"),
               annotation_text="5-min window",
               annotation_font_color="#1a7f37",annotation_font_size=10)

for x,lbl in [(50_000,"Gurobi OOM"),(200_000,"CLARABEL OOM")]:
    fig.add_vline(x=x,line=dict(color="#d1242f",width=1,dash="dot"))
    fig.add_annotation(x=x,y=0.9,text=lbl,showarrow=False,
                        font=dict(size=8,color="#d1242f",family="monospace"),
                        textangle=-90,yref="paper")

fig.update_layout(
    title=dict(text="Solve Time vs Fleet Size  <span style='font-size:11px;color:#1d4ed8'>[CACHED · PRISM internal benchmarks]</span>",
               font=dict(color="#1f2328",size=13),x=0.01),
    xaxis=dict(type="log",title="Fleet size (devices)",gridcolor="#e9ecef",zeroline=False,color="#57606a"),
    yaxis=dict(type="log",title="Solve time (s)",gridcolor="#e9ecef",color="#57606a"),
    legend=dict(bgcolor="#fff",bordercolor="#d0d7de",borderwidth=1),
    paper_bgcolor="#f6f8fa",plot_bgcolor="#f6f8fa",
    height=340,margin=dict(l=20,r=20,t=60,b=20),
    font=dict(family="monospace",color="#57606a"),
)
fig.show()

print(f"PRISM vs ADMM speedup: {BENCH['admm']['speedup'][0]}× (N=2k), {BENCH['admm']['speedup'][1]}× (N=10k)")
print(f"500k solve: {BENCH['gpu_scale']['prism_s'][2]}s  |  warm-start: {BENCH['warmstart']['warm_s']}s")


In [ ]:
# ── Section 5: Interactive — Adjust Parameters ───────────────────────────────
display(HTML(
    '<div class="pcard"><div class="stitle">Section 5 — Adjust Parameters</div>'
    '<p style="color:#57606a;font-size:13px;line-height:1.7;">'
    'Move the sliders and click <strong>Run Dispatch</strong> to recompute coordination '
    'premium and economics for your chosen scenario.'
    '</p></div>'
))

sl_n   = widgets.SelectionSlider(
    options=[("2,000",2000),("10,000",10000),("50,000",50000),
             ("200,000",200000),("500,000",500000)],
    value=10000, description="Fleet N:",
    style={"description_width":"80px"},layout=widgets.Layout(width="480px"),
)
sl_cap = widgets.IntSlider(
    min=30,max=100,step=5,value=60,
    description="Feeder cap %:",
    style={"description_width":"90px"},layout=widgets.Layout(width="480px"),
)
sl_eta = widgets.FloatSlider(
    min=0.75,max=0.95,step=0.05,value=0.85,readout_format=".2f",
    description="η (RTE):",
    style={"description_width":"80px"},layout=widgets.Layout(width="480px"),
)
sl_cyc = widgets.FloatSlider(
    min=0.5,max=2.5,step=0.5,value=1.5,readout_format=".1f",
    description="Cyc/day:",
    style={"description_width":"80px"},layout=widgets.Layout(width="480px"),
)
btn = widgets.Button(description="▶  Run Dispatch",
                     button_style="primary",
                     layout=widgets.Layout(width="200px",height="38px"))
out = widgets.Output()

def _interp_uplift(cap):
    pts = list(zip([30,45,60,100],[28.09,19.03,12.75,0.00]))
    if cap <= 30:  return 28.09
    if cap >= 100: return 0.00
    for i in range(len(pts)-1):
        if pts[i][0] <= cap <= pts[i+1][0]:
            t=(cap-pts[i][0])/(pts[i+1][0]-pts[i][0])
            return pts[i][1]+t*(pts[i+1][1]-pts[i][1])
    return 0.

PRISM_T = {2000:0.86,10000:0.72,50000:1.58,200000:6.24,500000:15.79}

def on_run(b):
    n   = sl_n.value
    cap = sl_cap.value
    eta = sl_eta.value
    cyc = sl_cyc.value
    uplift = _interp_uplift(cap)
    eta_f  = eta / 0.85
    cyc_f  = cyc / 1.5
    arb    = BENCH["economics"]["per_mw_yr"] * 1000 * eta_f * cyc_f
    prism$ = arb * uplift / 100
    t_s    = PRISM_T.get(n, 15.79)
    in_win = t_s < 300

    with out:
        clear_output(wait=True)
        display(HTML(
            f'<div class="pcard">'
            f'<div style="font-size:13px;font-weight:700;color:#1f2328;margin-bottom:12px;">'
            f'Results — N={n:,} · Cap={cap}% · η={eta:.2f} · {cyc} cyc/day</div>'
            f'<div class="mgrid">'
            f'<div class="mbox"><div class="mval">{uplift:.2f}%</div><div class="mlbl">Coordination premium</div></div>'
            f'<div class="mbox"><div class="mval" style="color:{"#1a7f37" if in_win else "#d1242f"}">{t_s:.2f}s</div>'
            f'<div class="mlbl">PRISM solve</div></div>'
            f'<div class="mbox"><div class="mval" style="color:{"#1a7f37" if in_win else "#d1242f"}">'
            f'{"IN ✓" if in_win else "OVER ✗"}</div><div class="mlbl">5-min window</div></div>'
            f'<div class="mbox"><div class="mval">${arb/1e6:.1f}M</div><div class="mlbl">Fleet arb./yr</div></div>'
            f'<div class="mbox"><div class="mval grn">${prism$/1e6:.1f}M</div><div class="mlbl">PRISM increment/yr</div></div>'
            f'<div class="mbox"><div class="mval">0.004%</div><div class="mlbl">Quality gap</div></div>'
            f'</div>'
            f'<p style="font-family:monospace;font-size:11px;color:#57606a;margin-top:8px;">'
            f'Economics = ${BENCH["economics"]["per_mw_yr"]:,}/MW/yr × 1,000 MW × η-factor × cyc-factor × {uplift:.2f}% uplift. '
            f'Capture rate assumption not included here.</p>'
            f'</div>'
        ))

btn.on_click(on_run)
on_run(None)  # show default on load
display(widgets.VBox([
    widgets.HBox([sl_n, sl_eta]),
    widgets.HBox([sl_cap, sl_cyc]),
    btn, out
]))


In [ ]:
# ── Section 6: Business Case [CACHED + ASSUMPTION] ───────────────────────────
scenarios = [
    ("Conservative", 2, 500,  12.75, 0.05),
    ("Base",         5, 800,  19.03, 0.10),
    ("Aggressive",   10,1000, 28.09, 0.15),
]
rows = ""
for sc, fleets, mw, uplift, cap in scenarios:
    arb  = BENCH["economics"]["per_mw_yr"]*mw*fleets/1e6
    inc  = arb*uplift/100
    arr  = inc*cap
    rows += (f'<tr><td style="font-weight:600">{sc}</td><td>{fleets}</td><td>{mw:,} MW</td>'
             f'<td class="grn">${arb:.1f}M</td><td class="grn">${inc:.1f}M</td>'
             f'<td>{cap*100:.0f}%</td><td class="blu">${arr:.1f}M</td></tr>')

display(HTML(
    '<div class="pcard"><div class="stitle">Section 6 — Scenario Analysis</div>'
    '<span class="bdg bdg-cached">Arbitrage: CACHED</span>&nbsp;'
    '<span class="bdg bdg-assum">Capture rate: ASSUMPTION</span>'
    '<div style="overflow-x:auto;margin-top:12px;">'
    '<table class="ptbl"><thead><tr>'
    '<th>Scenario</th><th>Fleets</th><th>Fleet MW</th>'
    '<th>Base Arb.</th><th>PRISM Uplift</th><th>Capture</th><th>ARR</th>'
    '</tr></thead><tbody>' + rows + '</tbody></table></div>'
    '<p style="color:#57606a;font-size:11px;font-family:monospace;margin-top:10px;">'
    'Base arbitrage: $61,140/MW/yr [CACHED · NYISO RT · η=0.85 · 1.5 cyc/day]<br>'
    'Capture rate = software licence as % of PRISM incremental value [ASSUMPTION]'
    '</p></div>'
))


In [ ]:
# ── Section 7: Warm-Start at 500k Units  [CACHED] ────────────────────────────
ws = BENCH["warmstart"]
display(HTML(
    '<div class="pcard"><div class="stitle">Section 7 — Warm-Start Cadence (N=500,000)</div>'
    '<span class="bdg bdg-cached">CACHED · PRISM warm-start benchmark</span>'
    '<table class="ptbl" style="margin-top:12px;"><thead><tr>'
    '<th>Cycle</th><th>Iterations</th><th>Solve time</th><th>5-min window</th><th>Feas. violation</th>'
    '</tr></thead><tbody>'
    f'<tr><td>Cycle 0 (cold)</td><td>{ws["cold_iters"]}</td>'
    f'<td class="amb">{ws["cold_s"]}s</td><td class="grn">YES</td><td class="grn">0.000</td></tr>'
    f'<tr><td>Cycle 1+ (warm)</td><td>{ws["warm_iters"]}</td>'
    f'<td class="grn">{ws["warm_s"]}s</td><td class="grn">YES</td><td class="grn">0.000</td></tr>'
    '</tbody></table>'
    f'<p style="color:#57606a;font-size:12px;font-family:monospace;margin-top:10px;">'
    f'Warm-start is {1-ws["warm_s"]/ws["cold_s"]:.0%} faster than cold. '
    f'Both within the 300-second clearing window. '
    f'Sustained real-time operation at 500k confirmed.</p></div>'
))


In [ ]:
# ── IP Safety Summary ─────────────────────────────────────────────────────────
display(HTML(
    '<div class="pcard"><div class="stitle">IP Safety Audit</div>'
    '<table class="ptbl"><thead><tr><th>Category</th><th>Status</th><th>Notes</th></tr></thead><tbody>'
    '<tr><td>Internal algorithm names</td><td class="grn">CLEAN</td><td>Not disclosed</td></tr>'
    '<tr><td>Solver update equations</td><td class="grn">CLEAN</td><td>Abstracted as "GPU-native proprietary engine"</td></tr>'
    '<tr><td>GPU / CUDA implementation</td><td class="grn">CLEAN</td><td>External timing only</td></tr>'
    '<tr><td>Internal tuning parameters</td><td class="grn">CLEAN</td><td>Not disclosed</td></tr>'
    '<tr><td>Benchmark number accuracy</td><td class="grn">CLEAN</td><td>All from validated files; labelled CACHED/SYNTHETIC/ASSUMPTION</td></tr>'
    '<tr><td>Coordination value claim</td><td class="grn">CLEAN</td><td>12–28% from coordination benchmark; methodology disclosed</td></tr>'
    '<tr><td>Economics claim</td><td class="grn">CLEAN</td><td>$16M defensible; ARR capture labelled ASSUMPTION</td></tr>'
    '</tbody></table>'
    '<div class="ipnote" style="margin-top:10px;">'
    'Safe to share with: seed investors, DERMS partners, VPP operator CTOs, energy-market strategy teams.<br>'
    'Production method available under NDA. Delivery: REST API, Docker, VPC, OEM SDK.'
    '</div></div>'
))
print("Demo complete. Contact: contact@asymmetrycomputing.com")
